In [ ]:
import os
import pandas as pd

def migrate_csv_to_parquet(base_path):
    print(f"🔍 {base_path} 내의 CSV 파일 변환을 시작합니다...")
    success_count = 0

    for root, dirs, files in os.walk(base_path):
        for file in files:
            if not file.endswith('.csv'): continue
            csv_path = os.path.join(root, file)
            parquet_path = csv_path.replace('.csv', '.parquet')

            if os.path.exists(parquet_path): continue
            if os.path.getsize(csv_path) == 0:
                print(f"⚠️ 빈 파일 건너뜀: {file}")
                continue

            try:
                with open(csv_path, 'r', encoding='cp949') as f:
                    df = pd.read_csv(f, sep='|', encoding='cp949')
                if df.empty: continue

                # PyArrow 충돌 시 fastparquet으로 우회
                try:
                    df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
                except Exception:
                    df.to_parquet(parquet_path, engine='fastparquet', compression='snappy')

                print(f"✅ 변환 완료: {file}")
                success_count += 1
            except Exception as e:
                print(f"❌ 실패 ({file}): {e}")

    print(f"✨ 완료! {success_count}개 변환 성공.")

migrate_csv_to_parquet("../../01_원본데이터/카드/3. 가맹점 정보(대민)(mer_s)")